In [44]:
import openeo

input_geojson = 'mystorage/high_negative_anomalies_wetlands.geojson'

In [45]:
import json
import glob
import os

res = {}
for json_path in glob.glob('mystorage/results/results*/lai_biopar_20*/timeseries.json'):
    with open(json_path, 'r') as f:
        json_res = json.load(f)
        if isinstance(json_res[0][0], float):
            wetland_id = int(json_path.split('/')[2][7:])
            year = int(os.path.dirname(json_path)[-4:])
            if wetland_id in res.keys():
                res[wetland_id].append(year)
            else:
                res[wetland_id] = [year]

print(res)

{103: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 11: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 142: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 157: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 17: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 18: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 28: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 3: [2015, 2016, 2017, 2018, 2019, 2020, 2021], 35: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 4: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 74: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]}


In [46]:
from itertools import product

with open(input_geojson, 'r', encoding='utf-8') as f:
    geojson_dict = json.load(f)

features = []
for feat in geojson_dict['features']:
    # print(feat['properties']['terrameter_id'], feat['geometry'])
    features.append(feat['properties']['terrameter_id'])
print(features)

years = range(2015, 2026)
expected_results = product(features, years)

[11, 103, 28, 4, 74, 18, 35, 3, 81, 59]


In [47]:
missing = []
for exp_id, exp_y in expected_results:
    if exp_id not in res.keys():
        missing.append((exp_id, exp_y))
    else:
        if exp_y not in res[exp_id]:
            missing.append((exp_id, exp_y))

print(missing)

[(3, 2022), (3, 2023), (3, 2024), (3, 2025), (81, 2015), (81, 2016), (81, 2017), (81, 2018), (81, 2019), (81, 2020), (81, 2021), (81, 2022), (81, 2023), (81, 2024), (81, 2025), (59, 2015), (59, 2016), (59, 2017), (59, 2018), (59, 2019), (59, 2020), (59, 2021), (59, 2022), (59, 2023), (59, 2024), (59, 2025)]


In [48]:
def request_LAI_calcul(wetland_id, year):
    # Create a processing graph from the BIOPAR process using an active openEO connection
    print()
    print('working for ID ' + str(wetland_id) + ' AND for year ' + str(y))

    # get the polygon shape
    with open(input_geojson, 'r', encoding='utf-8') as f:
        geojson_dict = json.load(f)
    for feat in geojson_dict['features']:
        if feat['properties']['terrameter_id'] == 18:
            my_poly = feat['geometry']
        
    lai_biopar = connection.datacube_from_process(
        "biopar", 
        namespace = "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/vito/biopar/openeo_udp/biopar.json",
        temporal_extent = [str(year) + '-07-01', str(year) + '-09-30'],
        spatial_extent= my_poly,
        biopar_type = 'LAI'
        )

    lai_biopar_singlet = lai_biopar.reduce_temporal(reducer='mean')
    aggregated = lai_biopar_singlet.aggregate_spatial(geometries=my_poly, reducer='mean')
    job = aggregated.create_job(out_format='JSON', title='CLMS_LAI_wetland_' + str(wetland_id) + '_Q3_' + str(year))
    job.start()
    return job

In [49]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

jobs = {}
for wet_id, y in missing:
    cur_job = request_LAI_calcul(wet_id, y)
    print('submitted job for wetland ' + str(wet_id) + ' and year ' + str(y) + ': id --> ' + cur_job.job_id)
    jobs[(wet_id, y)] = cur_job.job_id

Authenticated using refresh token.

working for ID 3 AND for year 2022
submitted job for wetland 3 and year 2022: id --> j-260514081523429e9ae6180b5e41821a

working for ID 3 AND for year 2023
submitted job for wetland 3 and year 2023: id --> j-260514081541459a9c1afe74dc960cac

working for ID 3 AND for year 2024
submitted job for wetland 3 and year 2024: id --> j-2605140815594ec8b242cd7a22e4a968

working for ID 3 AND for year 2025
submitted job for wetland 3 and year 2025: id --> j-26051408161843ec8341f7fcc03db16f

working for ID 81 AND for year 2015
submitted job for wetland 81 and year 2015: id --> j-260514081651413e92376d10e902d1bb

working for ID 81 AND for year 2016
submitted job for wetland 81 and year 2016: id --> j-2605140817064a0eb1fdeeed4d35d053

working for ID 81 AND for year 2017
submitted job for wetland 81 and year 2017: id --> j-260514081734436197fa140589270451

working for ID 81 AND for year 2018
submitted job for wetland 81 and year 2018: id --> j-2605140817584961a41de5

In [65]:
print(jobs)
print(len(jobs))

{(81, 2025): 'j-26051408201749a9887621e2e240eba8', (59, 2016): 'j-2605140821044944a2b620c327398fd4', (59, 2017): 'j-2605140821374a18b74084c22ec2453f', (59, 2018): 'j-26051408222048ce9d0e4226a88af49e', (59, 2019): 'j-2605140822454a8c8c381e3d3fa48b21', (59, 2020): 'j-26051408231742f5b0bd6c39f4524033', (59, 2021): 'j-260514082350412ab27c7465845e7c6e', (59, 2022): 'j-26051408240848849fb4a13450f9541b', (59, 2023): 'j-26051408242849609c0ae0752800822f', (59, 2024): 'j-26051408245549d3a5ee331a9be39dce', (59, 2025): 'j-260514082530492b9ce2434b61dfba59'}
11


In [66]:
jobs_to_delete = []
for key, job_id in jobs.items():
    job = connection.job(job_id)
    if job.status() == 'finished':
        results = job.get_results()
        results.download_files('mystorage/results/results' + str(key[0]) + '/lai_biopar_' + str(key[1]))
        print('results written for ' + str(key))
        jobs_to_delete.append(key)

results written for (81, 2025)
results written for (59, 2016)
results written for (59, 2017)
results written for (59, 2018)
results written for (59, 2019)
results written for (59, 2020)
results written for (59, 2021)
results written for (59, 2022)
results written for (59, 2023)
results written for (59, 2024)
results written for (59, 2025)


In [67]:
print(jobs_to_delete)

[(81, 2025), (59, 2016), (59, 2017), (59, 2018), (59, 2019), (59, 2020), (59, 2021), (59, 2022), (59, 2023), (59, 2024), (59, 2025)]


In [68]:
print(jobs)
for i in jobs_to_delete:
    del jobs[i]
print(jobs)

{(81, 2025): 'j-26051408201749a9887621e2e240eba8', (59, 2016): 'j-2605140821044944a2b620c327398fd4', (59, 2017): 'j-2605140821374a18b74084c22ec2453f', (59, 2018): 'j-26051408222048ce9d0e4226a88af49e', (59, 2019): 'j-2605140822454a8c8c381e3d3fa48b21', (59, 2020): 'j-26051408231742f5b0bd6c39f4524033', (59, 2021): 'j-260514082350412ab27c7465845e7c6e', (59, 2022): 'j-26051408240848849fb4a13450f9541b', (59, 2023): 'j-26051408242849609c0ae0752800822f', (59, 2024): 'j-26051408245549d3a5ee331a9be39dce', (59, 2025): 'j-260514082530492b9ce2434b61dfba59'}
{}
